# Lesson 19 · 🎨 Matrix transformations illustrated — watch a matrix move the whole of space: rotation, scaling, shear and "linear combinations of columns"

> The visual language comes from **"The Art of Linear Algebra"** (Hiranabe & Strang).

Use `laviz` to draw matrix/vector operations as pictures and build a spatial intuition for "matrix × vector = a linear combination of columns". Aurora's mel filter matrix `M @ stft` and DFT matrix `W @ x` are both direct instances of this picture.

**Learning goals**
1. Use `laviz`'s `matrix_4ways` to see four ways of reading the same matrix (as a whole / as numbers / as columns / as rows)
2. Tell apart the two vector × vector operations: the inner product (→ a scalar) and the outer product (→ a rank-1 matrix)
3. Use `mat_times_vec` to visualize matrix × vector = a linear combination of A's columns, and understand what the coefficients mean
4. Use `mat_times_mat_rank1` to split a matrix product into a sum of rank-1 matrices, building geometric intuition for SVD/LoRA
5. Map the mel filter matrix `M @ stft` and the DFT matrix `W @ x` onto the "linear combination of columns" picture

**Why it matters for Aurora**: the mel filter `M @ stft` and the DFT matrix `W @ x` are both direct instances of this picture; once you understand the geometric meaning of a linear combination, L21's audio processing with matrices has a concrete picture to stand on.

← **Previous lesson** · [L18 · Invertibility and rank](L18_invertibility.ipynb)

> Last lesson covered **invertibility and rank**: rank = information, null space = the directions that get crushed, and diagnosing singular matrices.
> This lesson explores **matrix transformations illustrated**.

## This lesson's story: setting space in motion (the eyes-on version of L12)

> **How to read this**: the algebra of matrix multiplication was written out in L12. This lesson is **mainly pictures** — what rotation, scaling and shear look like; the narration stays short, and you don't need to reread all of L12.

The geometric core of matrix multiplication fits in one sentence: keep your eye on where the basis vectors get sent, and every other vector moves along with them. This lesson draws that process with `laviz` — from "how many ways can you read a matrix" to "matrix × matrix = a sum of rank-1 matrices" — building up a visual language for transformations of space step by step.

## 0. Load the tools (consistent colors and layout)

## Notation first: look at the shapes, then the operations

Every object in linear algebra has a definite shape: a vector is `(n,)`, a matrix is `(m, n)`, and a matrix times a vector turns `(n,)` into `(m,)`. Each example marks the input, the transformation and the output first.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from laviz import (style, matrix_4ways, vec_times_vec,
                   mat_times_vec, mat_times_mat_rank1)
style()
print('laviz ready')

## Hands-on: for linear algebra objects, look at the shape first, then the meaning

Check each object's `.shape`: `v.shape` tells you the vector's dimension, and `A.shape` tells you how many input dimensions the matrix maps to how many output dimensions. Count them off against the code below, and you won't get the dimensions of a matrix product wrong.

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('vector length ||v|| =', np.linalg.norm(v))


## Code experiment: loop over a few vectors and watch how the matrix changes them

When the same matrix acts on different vectors, how do the direction and length of each change? Print the results side by side and look for the pattern.

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 1. Four ways to look at a matrix

As a whole / as mn numbers / as **n columns** / as **m rows**. Every operation that follows switches between these views.

Why start from the "columns"? Geometrically, matrix multiplication sends the basis vectors `[1,0]` and `[0,1]` each to a new position — and those two columns uniquely determine how every other vector is transformed. `laviz` shows the basis vectors before and after side by side, so the intuition "matrix = a change of coordinate system" is right there to see, with no formula to memorize.

In [ ]:
matrix_4ways(np.array([[1,2],[3,4],[5,6]]));

## Pause: what exactly are these four views?

Take the matrix `matrix_4ways` just drew as the example:

```
A = [[1, 2],
     [3, 4],
     [5, 6]]
```

This is a matrix with 3 rows and 2 columns (written 3×2). You can look at the same A in 4 completely different ways — just as with a single photo, you could call it "an image file", or "millions of pixels", or "red, green and blue channels stacked together". Different angles of description, but the same object.

1. **The whole-matrix view**: A is simply "an operation that maps 2-D space into 3-D space"; you don't need to care which numbers are inside.
2. **The mn-numbers view**: A holds 3×2=6 numbers in total, laid out one by one in a grid: `1, 2, 3, 4, 5, 6`. This is the most "rigid" view — numbers only, no structure.
3. **The n-columns view**: slice A vertically into 2 column vectors: column 1 is `[1, 3, 5]`, column 2 is `[2, 4, 6]`. Each of the two columns is an arrow in 3-D space.
4. **The m-rows view**: slice A horizontally into 3 row vectors: row 1 is `[1, 2]`, row 2 is `[3, 4]`, row 3 is `[5, 6]`. Each row is an arrow in 2-D space.

Why are they "all the same matrix"? Because the slicing differs, but not one of the 6 numbers inside has been added or removed — only the grouping has changed. Like cutting a cake: slice it across or slice it down and the pieces have different shapes, but the ingredients haven't changed.

**How will we actually switch between views later?** The "outer product" in Section 2 uses the "column × row" view (views 3 and 4 working together); "matrix × vector" in Section 3 uses the "n columns" view (view 3); the rank-1 decomposition in Section 4 is "column × row" again. Remember the names of these 4 views, and whenever we say "from the columns' point of view" or "from the rows' point of view", you'll know which way of slicing we mean.

## Interlude: what is "rank", and why is an outer product always "rank 1"?

L18 said "rank = information", but that's a bit abstract. Put another way: **rank = how many of the matrix's columns are "independent directions"**.

An analogy: you have a box of colored pens. If the box has red and blue pens, you can mix lots of colors (just blend red and blue in different proportions) — here there are 2 "independent colors". But if the box is full of "red pens in different shades", in the end they're all just different strengths of red — essentially only 1 independent color, and however you mix them you'll never get blue. A matrix's "rank" counts this kind of "independent direction": in a rank-2 matrix the two columns point different ways and span the whole plane; in a rank-1 matrix every column is just a multiple of the same direction, forever "flattened" onto one line.

Now look at the outer product. The outer product multiplies a column vector `u` (shape `(m,)`) by a row vector `vᵀ` (shape `(n,)`) to give an m×n matrix, following this pattern:

```
column k of u @ v.T = v[k] * u
```

In other words, **every column of the result is a multiple of the same vector `u`** (the multiple is the matching number in `v`). All the columns are squeezed onto the same direction — which is exactly the definition of "rank 1": only one independent direction. The code below checks this, and compares it with a rank-2 matrix so you can see the difference.

In [ ]:
import numpy as np

u = np.array([1.0, 2.0, 3.0])   # column vector (3,)
v = np.array([4.0, 5.0])        # the "contents" of the row vector (2,)

rank1_matrix = np.outer(u, v)   # 3x2 rank-1 matrix: the outer product of u and v
print('outer product u ⊗ v =')
print(rank1_matrix)
print('column 1 / column 2 =', rank1_matrix[:, 0] / rank1_matrix[:, 1], '  ← both columns are multiples of the same direction u')
print('rank of the matrix =', np.linalg.matrix_rank(rank1_matrix), '(rank 1: every column points the same way)')

print()
rank2_matrix = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # two columns pointing different ways
print('For comparison: a rank-2 matrix =')
print(rank2_matrix)
print('rank of the matrix =', np.linalg.matrix_rank(rank2_matrix), '(rank 2: the two columns point different ways and span a plane)')

## 2. Vector × vector — 2 kinds

- **(v1) Inner product**: row · column → a **scalar**
- **(v2) Outer product**: column · row → a **rank-1 matrix** ← the building block of every matrix decomposition

In [ ]:
vec_times_vec([1,2,3], [4,5,6]);

**🔗 Aurora**: inner product = the numerator of cosine similarity (recommendations); every DFT frequency bin is an inner product too. The outer product (rank 1) is the key link to SVD / LoRA.


## 3. Matrix × vector — 2 kinds (the most important picture)

- **(Mv1)** each row · the vector = a set of dot products
- **(Mv2)** read the result as **a linear combination of A's column vectors** — the soul of `Ax`.

In [ ]:
mat_times_vec(np.array([[1,2],[3,4],[5,6]]), [2,1]);

**✏️ Try it**: change the `[2,1]` above and watch how the result on the right (gold) changes with the "column-combination coefficients".

**🔗 Aurora**: `dft(x)` in `aurora/audio/transforms.py` is just `twiddle @ x` (line 64), where `twiddle[k,n] = exp(-2πjkn/N)` is the DFT matrix W; `mel_filterbank(...)` in `aurora/audio/mel.py` returns the filter matrix M, and `M @ stft_output` is a direct instance of the same picture applied to audio.

### Making it concrete: what does M @ stft_output actually look like?

We just said that the matrix `M` returned by `mel_filterbank(...)`, multiplied by the spectrogram `stft_output`, is also an instance of the "linear combination of columns" picture — but without concrete numbers, that sentence is still a bit up in the air. Below we build a "mini" example: shrink M to 2×3 (2 mel bands, 3 FFT bins) and `stft_output` to a vector of length 3, and actually compute `M @ stft_output`.

In the real Aurora code, **each column** of `M` corresponds to one FFT bin (say the finely spaced frequencies 100 Hz, 200 Hz, 300 Hz), and the numbers in that column are "how much this bin contributes to each mel band"; **each component** of `stft_output` is the energy in that bin for this frame of the signal. What `M @ stft_output` does is "distribute" each FFT bin's energy into the matching mel bands by weight, then add it all up — which is exactly "a linear combination of columns": use `stft_output` as the coefficients to weight M's columns, then sum.

In [ ]:
# A mini mel filter matrix: 2 mel bands x 3 FFT bins
M_mini = np.array([
    [0.5, 0.5, 0.0],   # mel band 1: takes in only the first two FFT bins, half weight each
    [0.0, 0.3, 0.7],   # mel band 2: leans toward the third FFT bin
])
stft_mini = np.array([2.0, 4.0, 6.0])  # energy in each of the 3 FFT bins (simplified to real numbers)

mel_out = M_mini @ stft_mini
print('M_mini @ stft_mini =', mel_out)

# Recompute it as a "linear combination of columns" and check the results agree
manual = (stft_mini[0] * M_mini[:, 0] + stft_mini[1] * M_mini[:, 1]
          + stft_mini[2] * M_mini[:, 2])
print('column combination by hand =', manual)

### Before writing code, fill in the variable table

Before writing `mat_times_vec`, be clear on three things:
- Input: a matrix `A` (shape `(m, n)`) and a coefficient vector `x` (length `n`)
- Key step: compute `A[:, k] * x[k]` to get each column's contribution, then add them up to get `Ax`
- Return: a figure of each column's contribution, laid out side by side to show "a linear combination of columns"

> **Note**: `mat_times_vec` is already implemented in the `laviz` helper; the above is a conceptual preview of how it works inside, and you're not asked to rewrite it. The part you actually do yourself is Section 5, "Exercise: implement `classify_transform(A)`".

### Optional deep dive: what is mat_times_vec actually computing inside?

`mat_times_vec` is already implemented in the `laviz` helper, so there's no need to rewrite it — but if you're curious what's "under the hood", the logic is quite plain: **multiply each column of the matrix by the matching number in the coefficient vector, then add these "scaled-up or scaled-down columns" together, tip to tail**. In everyday terms, it's like a buffet: each column of the matrix is a dish, the coefficient vector tells you "how many servings of this dish", and piling every serving onto the same plate gives you `A @ x`.

Below we write this logic out by hand (no laviz, just NumPy) and check the result against the same example from Section 3 — if you want to verify other examples by hand, use this code as a template and change the numbers.

In [ ]:
A_demo = np.array([[1, 2], [3, 4], [5, 6]])
x_demo = np.array([2, 1])

# Hand-made "matrix × vector = linear combination of columns"
total = np.zeros(A_demo.shape[0])
for k in range(A_demo.shape[1]):
    col_contribution = A_demo[:, k] * x_demo[k]
    total = total + col_contribution
    print(f'column {k} {A_demo[:, k]} × coefficient {x_demo[k]} = {col_contribution}')

print('added up      =', total)
print('directly with @ =', A_demo @ x_demo, '  ← the two should match')

## 4. Matrix × matrix = a sum of rank-1 matrices

`AB = Σ_k (column k of A) × (row k of B)`, and each term is a rank-1 matrix.

In [ ]:
mat_times_mat_rank1(np.array([[1,2],[3,4]]), np.array([[5,6],[7,8]]));

**🔗 Aurora**: this is the very same picture as SVD `A=Σσ·uvᵀ`, the spectral theorem `S=Σλ·qqᵀ`, and LoRA's low-rank updates. The next lesson (L20 · A map of decompositions) draws all five major decompositions.

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('How will matrix A move these vectors?')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | length {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


### Why does the rotation matrix look like that? Work backward from "where do the basis vectors turn to?"

Before moving on to the "parameter experiment", derive the rotation matrix formula yourself instead of memorizing it.

A matrix is determined by its "columns" — the rule from Section 1: **once you know where the basis vectors `[1,0]` and `[0,1]` are sent, the whole matrix is fixed** (because they are exactly the matrix's column 1 and column 2). So to build a matrix that "rotates the plane counterclockwise by an angle θ about the origin", you only need to answer two questions:

1. Where does `[1,0]` (the unit arrow pointing along the positive x-axis) land after turning by θ?
2. Where does `[0,1]` (the unit arrow pointing along the positive y-axis) land after turning by θ?

Think of `[1,0]` as a compass needle that starts out pointing at angle 0°. After turning by θ about the origin, its new coordinates come straight from trigonometry: `(cos θ, sin θ)` — exactly the coordinates of "the point at angle θ on the unit circle", a definition you've already met in high-school trigonometry. Likewise, `[0,1]` starts out pointing at 90°, and after turning by θ it points at `90°+θ`, with coordinates `(cos(90°+θ), sin(90°+θ)) = (-sin θ, cos θ)` (expand with the angle-addition formulas to check).

Put these two new coordinates side by side as two "columns", and you have the rotation matrix:

```
R(θ) = [[cos θ, -sin θ],
        [sin θ,  cos θ]]
        ↑column 1     ↑column 2
      ([1,0] turned) ([0,1] turned)
```

The code below checks it: turning by 90° should send `[1,0]` to `[0,1]` (the compass needle goes from pointing east to pointing north).

In [ ]:
theta_check = np.radians(90)
R_check = np.array([[np.cos(theta_check), -np.sin(theta_check)],
                     [np.sin(theta_check),  np.cos(theta_check)]])
print('R(90°) @ [1,0] =', np.round(R_check @ np.array([1.0, 0.0]), 6), '  ← should be about [0,1]')
print('R(90°) @ [0,1] =', np.round(R_check @ np.array([0.0, 1.0]), 6), '  ← should be about [-1,0]')

### By the way: why does the code use `np.radians(30)` instead of just writing 30?

Trigonometric functions like `np.cos` and `np.sin` follow the convention, in maths and in almost every programming language, that "the input angle is in radians, not degrees". A radian defines an angle as "arc length ÷ radius": one full turn (360°) is `2π` radians (about 6.28). If you hand `30` straight to `np.cos`, NumPy treats it as "30 radians" (a bit more than 4 full turns), not "30 degrees", and the rotation comes out completely wrong. `np.radians(30)` "converts 30 degrees to the matching value in radians" (`30 × π/180 ≈ 0.5236`), and only after that conversion is it right to feed it to `cos`/`sin`.

## Parameter experiment: use a rotation matrix to check that orthogonal matrices preserve distance

In the code for Section 3 (`mat_times_vec`), replace the matrix with a rotation matrix (θ = 30°):

```python
theta = np.radians(30)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
```

Watch the basis vectors `[1,0]` and `[0,1]` both rotate by 30°, while `np.linalg.norm(R @ v) == np.linalg.norm(v)` holds — a hands-on check that **orthogonal matrices preserve distance**. Compare the original matrix `[[2,1],[0,1]]`: it changes both direction and length (not orthogonal).

### Derive it by hand: why does R@Rᵀ=I mean "length doesn't change"?

Above we said "`R@Rᵀ=I` is a hands-on check that orthogonal matrices preserve distance"; here is the full derivation, with no steps skipped.

First recall: the squared length of a vector `v` can be written as "its inner product with itself": `‖v‖² = v · v = vᵀv` (each component times itself, all added up, which is exactly the definition of squared length — the higher-dimensional version of Pythagoras' theorem).

Now the vector is transformed by R into `Rv`, and its new squared length is:

```
‖Rv‖² = (Rv)ᵀ(Rv)
```

Transposes follow a rule: "the transpose of a product in brackets equals the product of the individual transposes in reverse order", so `(Rv)ᵀ = vᵀRᵀ`. Substituting:

```
‖Rv‖² = (Rv)ᵀ(Rv) = vᵀ Rᵀ R v
```

Here comes the key step: if R satisfies `R@Rᵀ=I` (an orthogonal matrix is defined by R and its transpose being each other's inverse, i.e. `RRᵀ=RᵀR=I`), then the `Rᵀ R` part is simply the identity matrix `I`:

```
‖Rv‖² = vᵀ (RᵀR) v = vᵀ I v = vᵀ v = ‖v‖²
```

Take square roots of both sides and you get `‖Rv‖ = ‖v‖` — the complete proof that "orthogonal matrices preserve distance": because `RᵀR` "cancels" to the identity matrix, the transformation in the middle has no effect on length at all. The assertion cell below checks this result once more with concrete numbers (`v=[3,4]`).

In [ ]:
# ✅ Actually visualize rotation / scaling / shear with laviz (delivering on the title's promise)
theta = np.radians(30)
R_30   = np.array([[np.cos(theta), -np.sin(theta)],   # rotate by 30°
                   [np.sin(theta),  np.cos(theta)]])
S_2    = np.array([[2., 0.], [0., 0.5]])              # stretch x by 2×, shrink y to 0.5×
K_1    = np.array([[1., 1.], [0., 1.]])              # horizontal shear k=1

v_unit = [1., 0.]
print('=== rotation matrix R(30°) × [1,0] ===')
mat_times_vec(R_30, v_unit)
print('=== scaling matrix S × [1,0] ===')
mat_times_vec(S_2, v_unit)
print('=== shear matrix K × [1,0] ===')
mat_times_vec(K_1, v_unit)


### What does det(A) actually measure? — area's "scale factor + whether it flips over"

Geometrically, the number `det` (the determinant) answers one question: **how many times bigger does this matrix make "area", and does it turn the shape "over"?**

Concretely: in 2D, take the square spanned by the basis vectors `[1,0]` and `[0,1]`, with area 1. Matrix A turns these two basis vectors into A's two columns, and those two new arrows span a parallelogram — the **absolute value** of `det(A)` is the area of that parallelogram (how many times the original square's area it is). The **sign** of `det(A)` marks whether "orientation" has been flipped: the basis vectors `[1,0]→[0,1]` start out arranged "counterclockwise", and if the two new columns are still arranged counterclockwise after the transformation (a pure rotation, for example), det is positive; if the transformation "flips" their relative order to clockwise (like a mirror), det is negative — like turning a sheet of paper with writing on it over to the back, where the writing becomes a mirror image. That "turning over" is exactly the geometric source of det's minus sign.

Take the scaling matrix `S = [[2,0],[0,0.5]]` from the visualization above as an example: det(S) = 2×0.5 = 1.0, meaning "area is first doubled, then halved, so overall the area is unchanged, and nothing is flipped". The reflection matrix `[[0,1],[1,0]]` (swaps x and y, the same as mirroring across the line y=x) has det = 0×0 − 1×1 = −1: the absolute value 1 says the area is unchanged, and the minus sign says it was "flipped over".

The code below runs through det for "rotation" and "reflection", compares their signs, then looks at an anisotropic scaling example to confirm that "the absolute value of det is the area scale factor".

In [ ]:
theta_demo = np.radians(40)
R_demo = np.array([[np.cos(theta_demo), -np.sin(theta_demo)],
                    [np.sin(theta_demo),  np.cos(theta_demo)]])
Refl_demo = np.array([[0., 1.], [1., 0.]])

print('det of rotation matrix R(40°) =', np.linalg.det(R_demo), '  (area unchanged, not flipped → positive)')
print('det of reflection matrix [[0,1],[1,0]] =', np.linalg.det(Refl_demo), '  (area unchanged, but flipped → negative)')

S_demo = np.array([[2., 0.], [0., 3.]])
print('det of scaling matrix [[2,0],[0,3]] =', np.linalg.det(S_demo), '  (area scaled up 2x3=6 times; that 6 is det)')

### "Preserving distance" and "preserving area" are two different things — rotation vs shear

It's easy to mix up "which properties a transformation preserves", so here the two ideas are pulled apart and compared:

- **Preserving distance** (lengths unchanged): only orthogonal matrices have this property (`R@Rᵀ=I`); rotations and reflections both belong here. We proved it above: `‖Rv‖=‖v‖`.
- **Preserving area** (absolute value of det = 1): only requires "the overall area scale factor is 1", not that every vector keeps its length. A shear matrix is the classic example.

Picture a deck of playing cards stacked neatly (a rectangular cross-section), then push the top sideways with your hand so the stack becomes a slanted parallelogram — no card rotates or changes shape, and the "area" the stack takes up doesn't change, but its outline goes from a rectangle to a slant, and the side lengths change too. That's a shear: area unchanged, but angles and lengths can both change.

The code below checks it: the shear matrix `K=[[1,1],[0,1]]` has det=1 (area-preserving), but `K@Kᵀ ≠ I` (not distance-preserving; vector lengths change).

In [ ]:
K_demo = np.array([[1., 1.], [0., 1.]])
v_demo = np.array([0.0, 1.0])

print('det of K =', np.linalg.det(K_demo), '  ← area unchanged (area-preserving)')
print('K @ Kᵀ =')
print(K_demo @ K_demo.T)
print('Equal to the identity matrix I?', np.allclose(K_demo @ K_demo.T, np.eye(2)), '  ← no, so a shear does not preserve distance')
print('‖v‖ =', np.linalg.norm(v_demo), '   ‖K@v‖ =', np.linalg.norm(K_demo @ v_demo), '  ← the length really did change')

In [ ]:
import numpy as np

# Assertions on the mathematical properties of geometric transformations (runs on its own; nothing for you to implement)
theta = np.radians(30)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
Refl = np.array([[0., 1.], [1., 0.]])
Scale = np.array([[2., 0.], [0., 0.5]])
Shear = np.array([[1., 1.], [0., 1.]])

# Rotation matrix: orthogonal + det=+1 + preserves length
assert np.allclose(R @ R.T, np.eye(2), atol=1e-12), "a rotation matrix should satisfy R@Rᵀ=I"
assert np.isclose(np.linalg.det(R), 1.0, atol=1e-12), "a rotation matrix should have det +1"
v = np.array([3., 4.])
assert np.isclose(np.linalg.norm(R @ v), np.linalg.norm(v), atol=1e-12), "rotation preserves vector length"
print(f"rotation matrix ✅  R@Rᵀ=I, det={np.linalg.det(R):.4f}, ||R@v||=||v||={np.linalg.norm(v):.4f}")

# Reflection matrix: orthogonal + det=-1
assert np.allclose(Refl @ Refl.T, np.eye(2), atol=1e-12), "a reflection matrix should satisfy Refl@Reflᵀ=I"
assert np.isclose(np.linalg.det(Refl), -1.0, atol=1e-12), "a reflection matrix should have det -1"
print(f"reflection matrix ✅  Refl@Reflᵀ=I, det={np.linalg.det(Refl):.4f}")

# Scaling matrix: diagonal + det = product of the diagonal entries
det_scale = np.linalg.det(Scale)
assert np.isclose(det_scale, Scale[0,0] * Scale[1,1], atol=1e-12), "scaling matrix det = product of the diagonal entries"
assert np.isclose(Scale[0,1], 0.0) and np.isclose(Scale[1,0], 0.0), "a scaling matrix is diagonal"
print(f"scaling matrix ✅  det={det_scale:.4f} = {Scale[0,0]}×{Scale[1,1]}, purely diagonal")

# Shear matrix: det=1 (area-preserving) but not orthogonal
assert np.isclose(np.linalg.det(Shear), 1.0, atol=1e-12), "a shear matrix preserves area: det=1"
assert not np.allclose(Shear @ Shear.T, np.eye(2), atol=1e-4), "a shear matrix is not orthogonal"
print(f"shear matrix ✅  det=1 (area-preserving), but K@Kᵀ≠I (changes angles)")
print("\nAll geometric transformation properties check out ✅")

## 5. ✏️ Exercise: implement `classify_transform(A)`

Given a 2×2 matrix, decide which geometric transformation it represents:

- **rotation**: `A @ Aᵀ ≈ I` and `det(A) ≈ +1`
- **reflection**: `A @ Aᵀ ≈ I` and `det(A) ≈ −1`
- **scaling**: a diagonal matrix (`A[0,1] ≈ A[1,0] ≈ 0`)
- **shear**: anything else

**Hint**: compute `np.linalg.det(A)` and `A @ A.T` first, then check the cases one by one.

**⚠️ Order matters**: you must check orthogonality first (step 1 rotation, step 2 reflection), and only then check for diagonal (step 3 scaling). If you check for diagonal first, `[[0,1],[1,0]]` can be misclassified — implement it strictly in the order above.

### See it for yourself: what happens when the order is wrong

Just saying "order matters" is rather abstract, so below we run both decision paths directly on the reflection matrix `[[0,1],[1,0]]` and see at which step the wrong order "misreads" it.

In [ ]:
M_test = np.array([[0., 1.], [1., 0.]])  # reflection matrix: a mirror across y=x

is_orthogonal = np.allclose(M_test @ M_test.T, np.eye(2))
is_diagonal = np.isclose(M_test[0, 1], 0.0) and np.isclose(M_test[1, 0], 0.0)
det_val = np.linalg.det(M_test)

print('M_test =')
print(M_test)
print('Orthogonal (M@Mᵀ≈I)?', is_orthogonal)
print('Diagonal (off-diagonal entries ≈0)?', is_diagonal)
print('det(M_test) =', det_val)

print()
print('--- ❌ Wrong order: check "diagonal?" first ---')
if is_diagonal:
    print('classified as scaling')
else:
    print('not a diagonal matrix → classified as shear   ← wrong! M_test is clearly a reflection, yet it lands in shear')

print()
print('--- ✅ Right order: check "orthogonal?" first ---')
if is_orthogonal and np.isclose(det_val, 1.0):
    print('classified as rotation')
elif is_orthogonal and np.isclose(det_val, -1.0):
    print('classified as reflection   ← right! Confirm orthogonality first, then det=-1 identifies it correctly as a reflection')
elif is_diagonal:
    print('classified as scaling')
else:
    print('classified as shear')

**Why does this happen?** In the matrix `[[0,1],[1,0]]`, the two numbers on the diagonal (top-left, bottom-right) are both 0, and the two numbers **off** the diagonal (top-right, bottom-left) are both 1. The "diagonal?" check looks precisely at "are the off-diagonal entries 0?", and this matrix is the exact opposite, so the "diagonal check" correctly decides it is "not a diagonal matrix" — and then (in the wrong order) it drops straight into the shear branch, skipping reflection, where it belongs. Only by doing the "orthogonality check" first can you use `det=-1` to identify it correctly as a reflection before it ever reaches the shear branch. That's why the exercise keeps stressing: **check orthogonality first (rotation/reflection), then diagonal (scaling), and only what's left counts as shear** — change the order, and a branch in the middle intercepts matrices that belong to an earlier category and drops them into the wrong bucket.

In [ ]:
import numpy as np

def classify_transform(A, tol=1e-9):
    """Return 'rotation' | 'reflection' | 'scaling' | 'shear'"""
    A = np.asarray(A, float)
    # ✏️ TODO: use det + AᵀA to decide the type of transformation
    raise NotImplementedError('Please implement classify_transform')

# Check
theta = np.pi / 3
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])   # rotate by 60°
Refl = np.array([[0., 1.], [1., 0.]])            # reflect across y=x (det=-1, not diagonal, won't be mistaken for scaling)
Scale = np.array([[2., 0.], [0., 3.]])            # anisotropic scaling
Shear = np.array([[1., 1.], [0., 1.]])            # shear

try:
    assert classify_transform(R) == 'rotation',    f'rotation matrix misclassified: {classify_transform(R)}'
    assert classify_transform(Refl) == 'reflection', f'reflection matrix misclassified: {classify_transform(Refl)}'
    assert classify_transform(Scale) == 'scaling',  f'scaling matrix misclassified: {classify_transform(Scale)}'
    assert classify_transform(Shear) == 'shear',    f'shear matrix misclassified: {classify_transform(Shear)}'
    print('✅ Passed: rotation/reflection/scaling/shear all classified correctly')
except (NotImplementedError, TypeError):
    print('⬆️  Implement classify_transform first, then run this')


## Wrap-up

You can now use `laviz` to draw `A @ x` as "A's columns times x's coefficients, added up", and draw `A @ B` as a stack of rank-1 matrices. The first is the column view of matrix × vector, the second the rank-1 decomposition view of matrix × matrix — and the two share one picture. In Aurora these two pictures correspond to the frequency mapping `mel_filterbank @ stft_output` and the spectrum computation of the DFT matrix `W @ x`. The next lesson (L20 · A map of decompositions) uses the same visual language to unfold SVD and eigendecomposition, and the "sum of rank-1 matrices" structure will be reused directly.

### One table: what the inner and outer products each do in Aurora

| | Input shape | Output | What it corresponds to in Aurora | Which operation it powers |
|---|---|---|---|---|
| **Inner product** (row·column→scalar) | two vectors of the same length | one number | the value of each DFT frequency bin = the inner product of the signal with a "probe wave"; the numerator of the cosine similarity between two vectors in a recommender system | the "each row · vector" algorithm for matrix × vector (the (Mv1) view in Section 3) |
| **Outer product** (column·row→rank-1 matrix) | one column vector + one row vector | one matrix (rank 1) | each term of the SVD `A=Σσ·uvᵀ`; LoRA approximates a weight update with a pair of "thin" matrices, which is also a stack of rank-1 (or low-rank) pieces at heart | the "sum of column×row" algorithm for matrix × matrix (Section 4) |

Put simply: **the inner product "compresses" — it boils a pile of numbers down to one** (measuring how alike two vectors are, or how strong one frequency component is); **the outer product "spreads out" — it lays two vectors out into a whole matrix** (building the "basic building blocks" of a decomposition). The matrix × vector picture (Section 3) uses the row-by-row version of the inner product; the matrix × matrix picture (Section 4) uses a term-by-term sum of outer products. In the next lesson (L20) you'll see that SVD splits a matrix into several "outer-product building blocks" stacked up, with each block's "worth" set by one singular value σ.

In [ ]:
# ✏️ Self-review for this lesson
l19_review = {
    "matvec_picture_understood": None,  # can explain the column-linear-combination picture of A@x? True/False
    "matmul_rank1_intuition":    None,  # understand the rank-1 stacking view of A@B? True/False
    "classify_transform_done":   None,  # classify_transform implemented and passes its asserts? True/False
    "transform_det_sign":        None,  # know that rotation has det=+1 and reflection det=-1? True/False
}

unfilled = [k for k, v in l19_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l19_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L19 complete! On to L20: a map of decompositions')


---

→ **Next lesson** · [L20 · A map of decompositions](L20_visual_factorizations.ipynb)

> Next lesson covers **a map of decompositions**: the geometric meaning of the LU / QR / SVD decompositions, compared.